# RetinaVessel AI — validation robustness and efficiency v3

This notebook evaluates the three verified seed-42 checkpoints on the frozen 89-image FIVES development-validation partition under the nine predeclared clean/degraded conditions. It records pixel metrics, clDice, thin-centerline recall, skeleton fragmentation, native-resolution latency, peak CUDA memory and image-bootstrap intervals.

**Protocol boundary:** attach the FIVES Kaggle dataset and either the v2 notebook output or its `retinavessel_v2_seed42.zip` artifact. This notebook must not inspect or score the official FIVES test images. Enable a GPU accelerator before running.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "f17833fdb25c9232e1df9cd60b0b5eb3ede048d6"
REPO = Path("/kaggle/working/retinavessel-ai-v3-code")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned evaluation code:", actual_commit)

In [ ]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest, samples_from_frozen_split

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v3"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
EXPECTED_SPLIT_SHA256 = "aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87"
assert hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = json.loads(SPLIT_MANIFEST.read_text())
_, validation = samples_from_frozen_split(load_manifest(FIVES_ROOT, "train"), split)
assert len(validation) == 89
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Frozen development validation images:", len(validation))
print("Official test images opened or evaluated: 0")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
EXTRACTED = Path("/kaggle/working/v2_verified_artifacts")
archives = sorted(INPUT_ROOT.rglob("retinavessel_v2_seed42.zip"))
if archives and not EXTRACTED.exists():
    EXTRACTED.mkdir(parents=True)
    shutil.unpack_archive(archives[0], EXTRACTED)
SEARCH_ROOTS = [INPUT_ROOT] + ([EXTRACTED] if EXTRACTED.exists() else [])

EXPECTED_CHECKPOINTS = {
    "unet_seed42": ("unet_seed42_es", "202a2d387663fd6e2e8db3463e79ce64fb72e278bc6b4655475a893b3f3162ad"),
    "lite_unet_seed42": ("lite_unet_seed42_es", "e1022147a4ed0f44c918ee694c53628687a3dbe687370ff79d30cf6908f63abb"),
    "lite_robust_seed42": ("lite_robust_seed42_es", "b48e12f1e17c7434f14b2adf51b2aac976b7a07b007049d7463eac34fa567395"),
}

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verified_checkpoint(run_directory, expected_sha256):
    candidates = []
    for root in SEARCH_ROOTS:
        candidates.extend(path for path in root.rglob("best.pt") if path.parent.name == run_directory)
    matches = [path for path in candidates if file_sha256(path) == expected_sha256]
    if not matches:
        raise FileNotFoundError(f"Attach verified v2 artifacts; missing {run_directory}/best.pt")
    return sorted(matches, key=lambda path: (len(path.parts), str(path)))[0]

CHECKPOINTS = {
    model_id: verified_checkpoint(run_directory, expected_sha256)
    for model_id, (run_directory, expected_sha256) in EXPECTED_CHECKPOINTS.items()
}
for model_id, checkpoint in CHECKPOINTS.items():
    print(model_id, checkpoint, file_sha256(checkpoint))

In [ ]:
manifest_candidates = []
for root in SEARCH_ROOTS:
    manifest_candidates.extend(root.rglob("run_manifest.json"))
verified_manifests = []
for path in manifest_candidates:
    payload = json.loads(path.read_text())
    if (payload.get("split_manifest_sha256") == EXPECTED_SPLIT_SHA256
            and payload.get("official_test_evaluated") is False
            and len(payload.get("results", [])) == 3):
        verified_manifests.append((path, payload))
assert verified_manifests, "No valid v2 run_manifest.json found in attached artifacts"
V2_MANIFEST_PATH, V2_MANIFEST = sorted(verified_manifests, key=lambda item: str(item[0]))[0]
assert V2_MANIFEST["training_code_commit"] == "0c40aa665be3a247372062eaa46a33f578cf7b0f"
assert V2_MANIFEST["maximum_epochs"] == 30
assert V2_MANIFEST["early_stopping_patience"] == 5
print("Verified v2 run manifest:", V2_MANIFEST_PATH)

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

## Full validation-only run

This executes 3 models × 9 conditions × 89 images = 2,403 native-resolution inferences. Do not add `--validation-limit`; that option is reserved for debugging. The exact severity and metric definitions are committed in `docs/v3_validation_protocol.md`.

In [ ]:
OUTPUT = REPO / "outputs" / "validation_v3_seed42"
command = [
    sys.executable, "-m", "retinavessel.validation",
    "--data", str(DATA_MOUNT),
    "--split-manifest", str(SPLIT_MANIFEST),
    "--expected-split-sha256", EXPECTED_SPLIT_SHA256,
    "--model", f"unet_seed42={CHECKPOINTS['unet_seed42']}",
    "--model", f"lite_unet_seed42={CHECKPOINTS['lite_unet_seed42']}",
    "--model", f"lite_robust_seed42={CHECKPOINTS['lite_robust_seed42']}",
    "--topology",
    "--device", "cuda",
    "--warmup-runs", "2",
    "--degradation-seed", "2026",
    "--bootstrap-seed", "2026",
    "--bootstrap-repetitions", "2000",
    "--out", str(OUTPUT),
]
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
print("Running full v3 evaluation; this is expected to take substantial GPU time.")
subprocess.run(command, cwd=REPO, env=evaluation_environment, check=True)

In [ ]:
import pandas as pd

report = json.loads((OUTPUT / "validation_report.json").read_text())
per_image = pd.read_csv(OUTPUT / "validation_per_image.csv")
summary = pd.read_csv(OUTPUT / "validation_summary.csv")
drops = pd.read_csv(OUTPUT / "paired_degradation_drops.csv")
assert report["official_test_evaluated"] is False
assert report["validation_image_count"] == 89
assert report["debug_validation_limit"] is None
assert len(per_image) == 3 * 9 * 89
assert len(summary) == 3 * 9
assert set(per_image["image_id"]) == set(split["validation"])
display(summary[["model_id", "condition", "mean_dice", "mean_cldice", "mean_thin_centerline_recall", "median_latency_ms", "maximum_peak_accelerator_bytes"]])
display(drops)
print("Official test evaluated:", report["official_test_evaluated"])

In [ ]:
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v3_validation_seed42")
archive = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=OUTPUT.parent, base_dir=OUTPUT.name))
print(f"Artifact archive: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Save this notebook version, then download the archive before starting multi-seed replication.")

## Stop here

Do not run the official FIVES test evaluation from this notebook. Review the complete validation artifacts, then run the separately versioned additional-seed training stage. Freeze the winning model and final analysis policy before opening the official test partition once.